# The following code cells illustrate how to use DynaTab for binary and multiclass classification, as well as regression, with or without Optuna-based hyperparameter tuning.

> **Citation.** If you use DynaTab in your work, please cite:  
> *Al Zadid Sultan Bin Habib, Gianfranco Doretto, and Donald A. Adjeroh.*  
> **DynaTab: Dynamic Feature Ordering as Neural Rewiring for High-Dimensional Tabular Data.**  
> In **AAAI 2026 First International Workshop on Neuro for AI & AI for Neuro: Towards Multi-Modal Natural Intelligence (NeuroAI) Workshop Proceedings (PMLR)**.
> Shorter citation: A. Z. S. B. Habib, G. Doretto, and D. A. Adjeroh, *“DynaTab: Dynamic Feature Ordering as Neural Rewiring for High-Dimensional Tabular Data,”* AAAI 2026 NeuroAI Workshop Proceedings (PMLR).

In [1]:
import importlib

def main():
    pkg = importlib.import_module("dynatab")
    print("dynatab imported OK")

    # Check that every name in dynatab.__all__ is actually accessible
    missing = []
    for name in getattr(pkg, "__all__", []):
        if not hasattr(pkg, name):
            missing.append(name)

    if missing:
        raise RuntimeError("Missing exports in dynatab: " + ", ".join(missing))

    print("All __all__ exports exist")
    print("Export count:", len(pkg.__all__))

if __name__ == "__main__":
    main()

dynatab imported OK
All __all__ exports exist
Export count: 29


In [4]:
from dynatab import (
    DynaTabBinary, DynaTabMulti, DynaTabRegression,
    TrainConfig, ModelConfig, LossConfig,
    DFOConfig, run_dfo, reorder_and_evaluate,
    train_one_split, evaluate_split, cross_validate,
    CustomFeatureLoss, evaluate_predictions,
    OrderAwarePositionalEmbedding, DynamicMaskedAttention, create_dma_mask,
)

In [3]:
from dynatab import DynaTabBinary, DynaTabMulti, DynaTabRegression
from dynatab import TrainConfig, ModelConfig, LossConfig
from dynatab import train_one_split, evaluate_split, cross_validate

In [4]:
from dynatab.model import DynaTabBinary, DynaTabMulti, DynaTabRegression
from dynatab.trainer import train_one_split, evaluate_split, cross_validate
from dynatab.dfo import DFOConfig, run_dfo, reorder_and_evaluate
from dynatab.customloss import CustomFeatureLoss  # alias works now

In [5]:
def check():
    from dynatab import DynaTabBinary, DFOConfig, TrainConfig, CustomFeatureLoss
    from dynatab import run_dfo, train_one_split
    print("key imports OK")

check()

key imports OK


In [3]:
import torch
from dynatab.model import DynaTabBinary

m = 10
model = DynaTabBinary(num_features=m, embedding_dim=128, backbone="Transformer")
xb = torch.randn(4, m)                 # [B,m]
go = torch.arange(m).long()            # [m]
imp = torch.randn(m)                   # [m]
y = model(xb, go, imp)
print(y.shape)                         # should be [4,1]

torch.Size([4, 1])


In [1]:
import torch
from dynatab.model import DynaTabBinary

m = 10
model = DynaTabBinary(num_features=m, embedding_dim=128, backbone="Transformer")

xb = torch.randn(4, m)      # [B,m]
go = torch.arange(m).long() # [m]
imp = torch.randn(m)        # [m]

y = model(xb, go, imp)
print(y.shape)              # should be [4,1]


torch.Size([4, 1])


# Binary Classification || GLI-85 Dataset || HDLSS || Sequential Processor backbone: LSTM || Demo run with less epochs

In [1]:
# ============================================================
# GLI-85 (HDLSS) - DFO computed ONCE, then 5x5 CV with LSTM
# Paste this whole cell in a notebook and run.
# Requires: dynatab package with patched SequentialProcessorBinary.forward
# ============================================================

import os, json
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold

from dynatab.dfo import DynamicFeatureOrdering, DFOConfig as DFOAlgoConfig
from dynatab.trainer import (
    TrainConfig, LossConfig, ModelConfig,
    build_model_from_config, build_loss,
    train_one_split
)

# -----------------------------
# Settings
# -----------------------------
CSV_PATH = "GLI-85_encoded.csv"
LABEL_COL = "label"

DFO_CACHE_DIR = "./dfo_cache"
os.makedirs(DFO_CACHE_DIR, exist_ok=True)
DFO_COLS_PATH = os.path.join(DFO_CACHE_DIR, "GLI85_dfo_cols.json")

# DFO params (compute once)
DFO_METRIC = "manhattan"
DFO_NUM_CLUSTERS = 2
DFO_ORDER = "ascending"      # "ascending" | "descending"
DFO_MUT_PROB = 0.0
DFO_TOL = 0.001
DFO_SEED = 42

# CV
N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

# Training
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 4          # 22k tokens => keep small
PRINT_EVERY = 10

# Model (LSTM)
EMBEDDING_DIM = 32      # OPE/PIGL dim; keep small for 22k
LSTM_KW = dict(
    hidden_dim=128,
    num_layers=1,
    dropout_rate=0.1,    # NOTE: LSTM dropout is only active if num_layers > 1
)

# Loss
LOSS_MODE = "DFO"       # "standard" | "dispersion" | "DFO"
LAMBDA_DISP = 0.0
LAMBDA_GLOBAL = 0.0

# -----------------------------
# Utils
# -----------------------------
def to_numpy_1d(x):
    if isinstance(x, (pd.Series, pd.DataFrame)):
        x = x.values
    x = np.asarray(x).reshape(-1)
    return x

def fold_standardize(X_tr_df, X_va_df):
    mu = X_tr_df.mean(axis=0)
    sd = X_tr_df.std(axis=0).replace(0, 1.0)
    return (X_tr_df - mu) / sd, (X_va_df - mu) / sd

def compute_binary_class_weights(y_train_01: np.ndarray) -> torch.Tensor:
    """
    Balanced weights for {0,1} -> Tensor[2] float32.
    """
    y = y_train_01.reshape(-1).astype(int)
    counts = np.bincount(y, minlength=2).astype(np.float64)
    counts[counts == 0] = 1.0
    w = (len(y) / (2.0 * counts))  # inverse freq scaled
    return torch.tensor(w, dtype=torch.float32)

# -----------------------------
# 0) Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# 1) Load
# -----------------------------
df = pd.read_csv(CSV_PATH)
if LABEL_COL not in df.columns:
    raise ValueError(f"Label col '{LABEL_COL}' not found in {CSV_PATH}. Columns: {list(df.columns)[:10]}...")

y = df[LABEL_COL].astype(int)
X = df.drop(columns=[LABEL_COL])
print("Loaded:", X.shape, "label counts:", y.value_counts().to_dict())

# -----------------------------
# 2) DFO compute ONCE (or load cache)
# -----------------------------
if os.path.exists(DFO_COLS_PATH):
    with open(DFO_COLS_PATH, "r") as f:
        dfo_cols = json.load(f)
    print(f"Loaded cached DFO cols: {DFO_COLS_PATH}  (len={len(dfo_cols)})")
else:
    print("Computing DFO once on FULL dataset (HDLSS: can be slow/heavy)...")
    dfo_algo = DynamicFeatureOrdering(
        config=DFOAlgoConfig(
            metric=DFO_METRIC,
            num_clusters=DFO_NUM_CLUSTERS,
            order=DFO_ORDER,
            mutation_prob=DFO_MUT_PROB,
            tolerance=DFO_TOL,
            seed=DFO_SEED,
        ),
        device=device,
    )

    # NOTE: DFO expects (n_samples, n_features). This is huge (85 x 22283).
    X_reordered, centroids, labels, global_order = dfo_algo.reorder_and_evaluate(
        X, y_train=y,
        metric=DFO_METRIC,
        num_clusters=DFO_NUM_CLUSTERS,
        order=DFO_ORDER,
        mutation_prob=DFO_MUT_PROB,
        tolerance=DFO_TOL,
    )
    dfo_cols = list(X_reordered.columns)
    with open(DFO_COLS_PATH, "w") as f:
        json.dump(dfo_cols, f)
    print(f"Saved DFO cols -> {DFO_COLS_PATH}  (len={len(dfo_cols)})")

# Verify and reorder X to DFO order
missing = [c for c in dfo_cols if c not in X.columns]
if missing:
    raise ValueError(f"Cached DFO columns not compatible with current CSV. Missing (first 10): {missing[:10]}")
X = X[dfo_cols].copy()
m = X.shape[1]
print("Reordered X to DFO order:", X.shape)

# Since X is already in DFO order, global_ordering is identity
global_ordering = torch.arange(m, dtype=torch.long, device=device)

# -----------------------------
# 3) Build configs (fixed across folds)
# -----------------------------
train_cfg = TrainConfig(
    epochs=EPOCHS,
    lr=LR,
    batch_size=BATCH_SIZE,
    print_every=PRINT_EVERY,
    device=device,
)

loss_cfg = LossConfig(
    loss_mode=LOSS_MODE,
    lambda_disp=LAMBDA_DISP,
    lambda_global=LAMBDA_GLOBAL,
)

model_cfg = ModelConfig(
    task="binary",
    backbone="LSTM",
    embedding_dim=EMBEDDING_DIM,
    backbone_kwargs=LSTM_KW,
)

# -----------------------------
# 4) 5x5 CV (NO DFO recompute)
# -----------------------------
rkf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
val_accs = []

split_id = 0
total_splits = N_SPLITS * N_REPEATS

for tr_idx, va_idx in rkf.split(X, y):
    split_id += 1

    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    X_va = X.iloc[va_idx]
    y_va = y.iloc[va_idx]

    # fold-only normalization (avoid leakage)
    X_tr_s, X_va_s = fold_standardize(X_tr, X_va)

    # tensors -> device
    X_tr_t = torch.tensor(X_tr_s.values, dtype=torch.float32, device=device)
    y_tr_t = torch.tensor(to_numpy_1d(y_tr), dtype=torch.float32, device=device).view(-1, 1)

    X_va_t = torch.tensor(X_va_s.values, dtype=torch.float32, device=device)
    y_va_t = torch.tensor(to_numpy_1d(y_va), dtype=torch.float32, device=device).view(-1, 1)

    # importance scores from TRAIN fold only (on reordered columns)
    importance_scores = torch.var(X_tr_t, dim=0).detach()  # [m] on device

    # class weights from TRAIN fold only
    cw = compute_binary_class_weights(to_numpy_1d(y_tr)).to(device)  # Tensor[2] on device

    # build model + loss
    model = build_model_from_config(model_cfg, num_features=m).to(device)

    loss_fn = build_loss(
        task="binary",
        loss_cfg=loss_cfg,
        class_weights=cw,
        global_ordering_weights=None,
    ).to(device)

    # train (returns val metrics because we provide X_val/y_val)
    metrics = train_one_split(
        model=model,
        task="binary",
        X_train=X_tr_t,
        y_train=y_tr_t,
        global_ordering=global_ordering,
        importance_scores=importance_scores,
        loss_fn=loss_fn,
        class_weights=cw,
        X_val=X_va_t,
        y_val=y_va_t,
        eval_metrics=["acc"],
        cfg=train_cfg,
        loss_mode=loss_cfg.loss_mode,
    )

    acc = float(metrics.get("acc", np.nan))
    val_accs.append(acc)
    print(f"[Split {split_id:02d}/{total_splits}] val_acc={acc:.4f}")

print("\nDONE")
print(f"5x5 CV Val Accuracy: {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")

Device: cuda
Loaded: (85, 22283) label counts: {1: 59, 0: 26}
Loaded cached DFO cols: ./dfo_cache\GLI85_dfo_cols.json  (len=22283)
Reordered X to DFO order: (85, 22283)
Epoch 001 | loss=0.696618
  val: {'acc': 0.5294}
Epoch 010 | loss=0.676300
  val: {'acc': 0.5882}
Epoch 020 | loss=0.602951
  val: {'acc': 0.5294}
Epoch 030 | loss=0.485063
  val: {'acc': 0.7059}
Epoch 040 | loss=0.260982
  val: {'acc': 0.6471}
Epoch 050 | loss=0.204521
  val: {'acc': 0.5882}
[Split 01/25] val_acc=0.5882
Epoch 001 | loss=0.697715
  val: {'acc': 0.2941}
Epoch 010 | loss=0.687710
  val: {'acc': 0.5294}
Epoch 020 | loss=0.676803
  val: {'acc': 0.6471}
Epoch 030 | loss=0.669962
  val: {'acc': 0.6471}
Epoch 040 | loss=0.616233
  val: {'acc': 0.7059}
Epoch 050 | loss=0.690404
  val: {'acc': 0.4706}
[Split 02/25] val_acc=0.4706
Epoch 001 | loss=0.696209
  val: {'acc': 0.2941}
Epoch 010 | loss=0.692086
  val: {'acc': 0.6471}
Epoch 020 | loss=0.669516
  val: {'acc': 0.4118}
Epoch 030 | loss=0.620786
  val: {'acc

# Binary Classification || GLI-85 Dataset || HDLSS || Sequrntial Processor backbone: Mamba || Demo run with less epochs

In [3]:
import os, json
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from sklearn.model_selection import RepeatedStratifiedKFold

from dynatab.trainer import (
    TrainConfig, LossConfig, ModelConfig,
    build_model_from_config, build_loss,
    train_one_split,
)
from dynatab.dfo import DynamicFeatureOrdering, DFOConfig as DFOAlgoConfig


# -----------------------------
# Helpers
# -----------------------------
def fold_standardize(X_tr_df: pd.DataFrame, X_va_df: pd.DataFrame):
    mu = X_tr_df.mean(axis=0)
    sd = X_tr_df.std(axis=0).replace(0, 1.0)
    return (X_tr_df - mu) / sd, (X_va_df - mu) / sd

def to_numpy_1d(x):
    if isinstance(x, (pd.Series, pd.DataFrame)):
        x = x.values
    return np.asarray(x).reshape(-1)

def compute_binary_class_weights(y01: np.ndarray) -> torch.Tensor:
    y = y01.reshape(-1).astype(int)
    counts = np.bincount(y, minlength=2).astype(np.float64)
    counts[counts == 0] = 1.0
    w = (len(y) / (2.0 * counts))
    return torch.tensor(w, dtype=torch.float32)


# -----------------------------
# Runner
# -----------------------------
@dataclass
class HDLSSCVConfig:
    csv_path: str = "GLI-85_encoded.csv"
    label_col: str = "label"

    # cache
    cache_dir: str = "./dfo_cache"
    dfo_cols_filename: str = "GLI85_dfo_cols.json"

    # DFO (computed once)
    dfo_metric: str = "manhattan"
    dfo_num_clusters: int = 2
    dfo_order: str = "ascending"
    dfo_mut_prob: float = 0.0
    dfo_tol: float = 0.001
    dfo_seed: int = 42

    # CV
    n_splits: int = 5
    n_repeats: int = 5
    random_state: int = 42

    # Training
    epochs: int = 20
    lr: float = 1e-3
    batch_size: int = 2
    print_every: int = 5

    # Model: current portable Mamba
    embedding_dim: int = 16     # OPE/PIGL dim (keep tiny for 22k tokens)
    mamba_kwargs: dict = None   # filled in __post_init__

    # Loss
    loss_mode: str = "standard"  # start with standard for HDLSS sanity check
    lambda_disp: float = 0.0
    lambda_global: float = 0.0


class GLI85Mamba5x5CV:
    def __init__(self, cfg: HDLSSCVConfig):
        self.cfg = cfg
        os.makedirs(cfg.cache_dir, exist_ok=True)
        self.dfo_cols_path = os.path.join(cfg.cache_dir, cfg.dfo_cols_filename)

        if self.cfg.mamba_kwargs is None:
            # IMPORTANT: keep these small for loop-based scan
            self.cfg.mamba_kwargs = dict(
                d_model=64,
                n_layers=1,
                d_state=8,
                conv_kernel=4,
                dropout=0.1,
            )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _load_data(self):
        df = pd.read_csv(self.cfg.csv_path)
        if self.cfg.label_col not in df.columns:
            raise ValueError(f"Label col '{self.cfg.label_col}' not found. Columns: {list(df.columns)[:10]}...")

        y = df[self.cfg.label_col].astype(int)
        X = df.drop(columns=[self.cfg.label_col])
        return X, y

    def _get_or_compute_dfo_cols(self, X: pd.DataFrame, y: pd.Series):
        if os.path.exists(self.dfo_cols_path):
            with open(self.dfo_cols_path, "r") as f:
                dfo_cols = json.load(f)
            return dfo_cols, True

        # WARNING: computing DFO on 85x22283 can be heavy; cache is recommended.
        dfo_algo = DynamicFeatureOrdering(
            config=DFOAlgoConfig(
                metric=self.cfg.dfo_metric,
                num_clusters=self.cfg.dfo_num_clusters,
                order=self.cfg.dfo_order,
                mutation_prob=self.cfg.dfo_mut_prob,
                tolerance=self.cfg.dfo_tol,
                seed=self.cfg.dfo_seed,
            ),
            device=self.device,
        )

        X_reordered, centroids, labels, global_order = dfo_algo.reorder_and_evaluate(
            X, y_train=y,
            metric=self.cfg.dfo_metric,
            num_clusters=self.cfg.dfo_num_clusters,
            order=self.cfg.dfo_order,
            mutation_prob=self.cfg.dfo_mut_prob,
            tolerance=self.cfg.dfo_tol,
        )

        dfo_cols = list(X_reordered.columns)
        with open(self.dfo_cols_path, "w") as f:
            json.dump(dfo_cols, f)
        return dfo_cols, False

    def fit(self):
        print("Device:", self.device)

        # 1) Load
        X, y = self._load_data()
        print("Loaded:", X.shape, "label counts:", y.value_counts().to_dict())

        # 2) DFO cols once
        dfo_cols, loaded_cache = self._get_or_compute_dfo_cols(X, y)
        print(("Loaded cached" if loaded_cache else "Computed") + f" DFO cols: {self.dfo_cols_path} (len={len(dfo_cols)})")

        # 3) Apply DFO order to X
        missing = [c for c in dfo_cols if c not in X.columns]
        if missing:
            raise ValueError(f"Cached DFO columns not compatible with current CSV. Missing (first 10): {missing[:10]}")
        X = X[dfo_cols].copy()
        m = X.shape[1]
        print("Reordered X:", X.shape)

        # 4) global_ordering is identity because X already reordered to the final order
        global_ordering = torch.arange(m, dtype=torch.long, device=self.device)

        # 5) configs
        train_cfg = TrainConfig(
            epochs=self.cfg.epochs,
            lr=self.cfg.lr,
            batch_size=self.cfg.batch_size,
            print_every=self.cfg.print_every,
            device=self.device,
        )
        loss_cfg = LossConfig(
            loss_mode=self.cfg.loss_mode,
            lambda_disp=self.cfg.lambda_disp,
            lambda_global=self.cfg.lambda_global,
        )
        model_cfg = ModelConfig(
            task="binary",
            backbone="Mamba",
            embedding_dim=self.cfg.embedding_dim,
            backbone_kwargs=self.cfg.mamba_kwargs,
        )

        # 6) CV
        rkf = RepeatedStratifiedKFold(
            n_splits=self.cfg.n_splits,
            n_repeats=self.cfg.n_repeats,
            random_state=self.cfg.random_state,
        )
        total_splits = self.cfg.n_splits * self.cfg.n_repeats

        val_accs = []
        fold_models = []  # optional: store trained models (big). You can turn this off.

        split_id = 0
        for tr_idx, va_idx in rkf.split(X, y):
            split_id += 1
            print(f"\n=== Split {split_id:02d}/{total_splits} ===")

            X_tr = X.iloc[tr_idx]
            y_tr = y.iloc[tr_idx]
            X_va = X.iloc[va_idx]
            y_va = y.iloc[va_idx]

            # fold-only normalization
            X_tr_s, X_va_s = fold_standardize(X_tr, X_va)

            # tensors
            X_tr_t = torch.tensor(X_tr_s.values, dtype=torch.float32, device=self.device)
            y_tr_t = torch.tensor(to_numpy_1d(y_tr), dtype=torch.float32, device=self.device).view(-1, 1)

            X_va_t = torch.tensor(X_va_s.values, dtype=torch.float32, device=self.device)
            y_va_t = torch.tensor(to_numpy_1d(y_va), dtype=torch.float32, device=self.device).view(-1, 1)

            # fold-only importance
            importance_scores = torch.var(X_tr_t, dim=0).detach()

            # fold-only class weights
            cw = compute_binary_class_weights(to_numpy_1d(y_tr)).to(self.device)

            # build model + loss
            model = build_model_from_config(model_cfg, num_features=m).to(self.device)
            loss_fn = build_loss(
                task="binary",
                loss_cfg=loss_cfg,
                class_weights=cw,
                global_ordering_weights=None,
            ).to(self.device)

            # train
            metrics = train_one_split(
                model=model,
                task="binary",
                X_train=X_tr_t,
                y_train=y_tr_t,
                global_ordering=global_ordering,
                importance_scores=importance_scores,
                loss_fn=loss_fn,
                class_weights=cw,
                X_val=X_va_t,
                y_val=y_va_t,
                eval_metrics=["acc"],
                cfg=train_cfg,
                loss_mode=loss_cfg.loss_mode,
            )

            acc = float(metrics.get("acc", np.nan))
            val_accs.append(acc)
            print(f"[Split {split_id:02d}/{total_splits}] val_acc={acc:.4f}")

            fold_models.append(model)  # optional; remove if you don't want to store them

        out = {
            "val_accs": val_accs,
            "mean_val_acc": float(np.mean(val_accs)),
            "std_val_acc": float(np.std(val_accs)),
            "dfo_cols_path": self.dfo_cols_path,
            "m": m,
            "device": str(self.device),
            "cfg": self.cfg,
            "fold_models": fold_models,  # optional
        }

        print("\nDONE")
        print(f"5x5 CV Val Accuracy: {out['mean_val_acc']:.4f} ± {out['std_val_acc']:.4f}")
        return out


# -----------------------------
# Run
# -----------------------------
cfg = HDLSSCVConfig(
    csv_path="GLI-85_encoded.csv",
    label_col="label",

    # keep it realistic first
    epochs=10,
    batch_size=1,
    print_every=1,

    embedding_dim=16,
    mamba_kwargs=dict(
        d_model=64,
        n_layers=1,
        d_state=8,
        conv_kernel=4,
        dropout=0.1,
    ),

    # start standard; once it runs, you can flip to DFO loss
    loss_mode="standard",
)

runner = GLI85Mamba5x5CV(cfg)
results = runner.fit()

Device: cuda
Loaded: (85, 22283) label counts: {1: 59, 0: 26}
Loaded cached DFO cols: ./dfo_cache\GLI85_dfo_cols.json (len=22283)
Reordered X: (85, 22283)

=== Split 01/25 ===
Epoch 001 | loss=0.710904
  val: {'acc': 0.6471}
Epoch 002 | loss=0.703631
  val: {'acc': 0.6471}
Epoch 003 | loss=0.702243
  val: {'acc': 0.3529}
Epoch 004 | loss=0.700635
  val: {'acc': 0.3529}
Epoch 005 | loss=0.699393
  val: {'acc': 0.3529}
Epoch 006 | loss=0.713066
  val: {'acc': 0.6471}
Epoch 007 | loss=0.697795
  val: {'acc': 0.3529}
Epoch 008 | loss=0.701074
  val: {'acc': 0.6471}
Epoch 009 | loss=0.698999
  val: {'acc': 0.4118}
Epoch 010 | loss=0.696144
  val: {'acc': 0.7059}
[Split 01/25] val_acc=0.7059

=== Split 02/25 ===
Epoch 001 | loss=0.706672
  val: {'acc': 0.7059}
Epoch 002 | loss=0.703060
  val: {'acc': 0.7059}
Epoch 003 | loss=0.704419
  val: {'acc': 0.7059}
Epoch 004 | loss=0.697874
  val: {'acc': 0.2941}
Epoch 005 | loss=0.702061
  val: {'acc': 0.2941}
Epoch 006 | loss=0.700922
  val: {'acc'